# YELLOW RAW DATA ANALYSIS

In [52]:
import findspark
findspark.init()
import pyspark  
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampNTZType,
    IntegerType, DoubleType, LongType
)
from pyspark.sql import functions as f
from pathlib import Path
import pandas as pd

In [53]:
projectRoot = Path.cwd().parents[2]

ZONE_CSV = Path(projectRoot)/ "data" / "raw" / "tlc_trip_record" / "taxi_zone_lookup.csv"
PATH_PARQUETS = Path(projectRoot) / "data" / "raw" / "tlc_trip_record" / "green"
OUTPUT = Path(projectRoot) / "data" / "clean" / "clean_tlc_trip_record" / "green"

MIN_DATETIME = "2020-01-01"
MAX_DATETIME =  "2026-01-01"
MIN_YEAR = 2020
MAX_YEAR =  2025

MAX_TRIP_MILES = 50 
MIN_TRIP_MILES = 0.1
MAX_TRIP_TIME = 10_800
MIN_TRIP_TIME = 60

MIN_FARE = 1.0
MAX_FARE = 500.00

UNKNOWN_LOCATIONS = [264, 265]
CBD_CORE_ZONES = [161, 162, 163, 164, 186, 229, 230, 231]
AIRPORT_ZONES = [1, 132, 138]

In [54]:
projectRoot

PosixPath('/Users/jiayi/Universidad/Tercero/2-cuatrimestre/PD2/Museekar')

# Init Spark

In [55]:
def stop(spark):
    spark.sparkContext.stop()

spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [56]:
spark.sparkContext

<SparkContext master=local[6] appName=pyspark-shell>

# Raw Data

**Yellow Taxi Trip Data - Variable Descriptions**

| Variable Name | Python Type | Description |
| :--- | :--- | :--- |
| **VendorID** | `int` | Code for the taxi technology provider. |
| **tpep_pickup_datetime** | `datetime` | Date and time when the taxi meter was turned on. |
| **tpep_dropoff_datetime** | `datetime` | Date and time when the taxi meter was turned off. |
| **passenger_count** | `int` | Number of passengers in the taxi. |
| **trip_distance** | `float` | Trip distance in miles as reported by the taximeter. |
| **RatecodeID** | `int` | Final rate code applied at the end of the trip (e.g., standard, airport). |
| **store_and_fwd_flag** | `str` | Indicates if trip data was stored in vehicle memory before transmission ('Y' or 'N'). |
| **PULocationID** | `int` | Taxi Zone ID where the passenger was picked up. |
| **DOLocationID** | `int` | Taxi Zone ID where the passenger was dropped off. |
| **payment_type** | `int` | Code for the payment method used (e.g., credit card, cash). |
| **fare_amount** | `float` | Time-and-distance fare calculated by the meter. |
| **extra** | `float` | Miscellaneous extras and surcharges. |
| **mta_tax** | `float` | Automatically triggered tax based on the metered rate. |
| **tip_amount** | `float` | Tip amount (credit card tips only; cash tips not included). |
| **tolls_amount** | `float` | Total amount of all tolls paid during the trip. |
| **improvement_surcharge** | `float` | Surcharge assessed at the start of the trip. |
| **total_amount** | `float` | Total amount charged to the passenger (excluding cash tips). |
| **congestion_surcharge** | `float` | Amount collected for NYS congestion surcharge. |
| **airport_fee** | `float` | Fee for pickups at LaGuardia or JFK airports. |
| **cbd_congestion_fee** | `float` | Per-trip charge for the MTA's Congestion Relief Zone (effective Jan 5, 2025). |

Prior to analysis, we should do some kind of features selection, we need to identify the most relevant variables. This step is crucial since it reduces dimensionality of the dataset.

### **Selected Features - NYC Yellow Taxi Trip Records**

#### Trip Timestamps
- **`tpep_pickup_datetime`**  
  The exact date and time when the trip started (meter engaged).

- **`tpep_dropoff_datetime`**  
  The exact date and time when the trip ended (meter disengaged).

#### Trip Route & Distance
- **`trip_distance`**  
  The total distance traveled during the trip, measured in miles.

- **`PULocationID`**  
  The Taxi Zone ID where the passenger was picked up.

- **`DOLocationID`**  
  The Taxi Zone ID where the passenger was dropped off.

#### Fare & Charges
- **`tip_amount`**  
  The amount of tip given by the passenger (credit card tips only).

- **`total_amount`**  
  The final total amount charged to the passenger.

- **`congestion_surcharge`**  
  Additional charge for trips within New York State congestion zones.

- **`airport_fee`**  
  A fee applied for pickups at LaGuardia or John F. Kennedy airports.

## Load Data

In [57]:
def load_yellow(spark: SparkSession) -> DataFrame:
    """ 
        Load parquet 
    """
    schema = StructType([
        StructField("lpep_pickup_datetime",     TimestampNTZType()),
        StructField("lpep_dropoff_datetime",    TimestampNTZType()),
        StructField("trip_distance",            DoubleType()),
        StructField("PULocationID",             LongType()),
        StructField("DOLocationID",             LongType()),
        StructField("tip_amount",               DoubleType()),
        StructField("total_amount",             DoubleType()),
        StructField("congestion_surcharge",     DoubleType()),
        StructField("cbd_congestion_fee",       DoubleType())
    ])

    path = str(PATH_PARQUETS / "*.parquet")
    df = spark.read.schema(schema).parquet(path)

    df = df.withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')
    df = df.withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime')
    return df

def load_zones(spark: SparkSession) -> DataFrame:
    path = str(ZONE_CSV)
    df = spark.read.option("header", "true").csv(path)
    df = df.select(
        f.col("LocationID").cast(IntegerType()).alias("LocationID"),
        f.col("Borough").alias("Borough"),
        f.col("Zone").alias("Zone"),
        f.col("service_zone").alias("service_zone"),
    )
    return df

In [58]:
raw_data = load_yellow(spark)

26/02/14 17:45:50 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/jiayi/Universidad/Tercero/2-cuatrimestre/PD2/Museekar/data/raw/tlc_trip_record/green/*.parquet.
java.io.FileNotFoundException: File /Users/jiayi/Universidad/Tercero/2-cuatrimestre/PD2/Museekar/data/raw/tlc_trip_record/green/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

Although there are warnings that seem to be related to the path of the raw data, it loads all the data without any problem. Suppose we can just ignore them.

In [59]:
zones = load_zones(spark)

## Basic Analysis: Stats, Nulls, Outliers

### Zones

In [60]:
zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [61]:
zones.show(3)

+----------+-------+--------------------+------------+
|LocationID|Borough|                Zone|service_zone|
+----------+-------+--------------------+------------+
|         1|    EWR|      Newark Airport|         EWR|
|         2| Queens|         Jamaica Bay|   Boro Zone|
|         3|  Bronx|Allerton/Pelham G...|   Boro Zone|
+----------+-------+--------------------+------------+
only showing top 3 rows


### Trips

In [62]:
raw_data.show(5)
total_rows = raw_data.count()
print(f"Shape: {total_rows} rows, {len(raw_data.columns)} columns")

+-------------------+-------------------+-------------+------------+------------+----------+------------+--------------------+------------------+
|    pickup_datetime|   dropoff_datetime|trip_distance|PULocationID|DOLocationID|tip_amount|total_amount|congestion_surcharge|cbd_congestion_fee|
+-------------------+-------------------+-------------+------------+------------+----------+------------+--------------------+------------------+
|2019-12-18 15:52:30|2019-12-18 15:54:39|          0.0|         264|         264|      0.01|        4.81|                 0.0|              NULL|
|2020-01-01 00:45:58|2020-01-01 00:56:39|         1.28|          66|          65|      4.06|       24.36|                 0.0|              NULL|
|2020-01-01 00:41:38|2020-01-01 00:52:49|         2.47|         181|         228|      3.54|       15.34|                 0.0|              NULL|
|2020-01-01 00:52:46|2020-01-01 01:14:21|          6.3|         129|         263|       0.0|       25.05|                2.7

In [63]:
raw_data.printSchema()

root
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [64]:
num_cols = ["trip_distance", "tip_amount", "total_amount",
            "congestion_surcharge", "cbd_congestion_fee"]


#### Summary

In [65]:
raw_summary = raw_data.select(num_cols).summary().toPandas()

In [66]:
raw_summary

,summary,trip_distance,tip_amount,total_amount,congestion_surcharge,cbd_congestion_fee
0,count,5633750,5633750,5633750,4478782,539315
1,mean,52.34411409451974,1.7903166310185026,22.237948428788535,0.6817982545254491,0.07026830331068114
2,stddev,2373.4929100314607,2.969153388877785,17.85173719178906,1.187336594828692,0.21861106874862613
3,min,-33.69,-100.0,-501.0,-2.75,-0.75
4,25%,1.18,0.0,11.41,0.0,0.0
5,50%,2.12,0.76,17.8,0.0,0.0
6,75%,4.21,2.75,27.18,0.0,0.0
7,max,360068.14,641.2,4004.5,2.75,0.75


### Nulls

In [67]:
def count_nulls(df:DataFrame):
    null_counts = {}
    for col in df.columns:
        nulls = df.where(f.col(col).isNull()).count()
        null_counts[col] = nulls

    for col, n in null_counts.items():
        percent = (n / total_rows * 100) if total_rows > 0 else 0
        print(f"{col}: {n:,} ({percent:.2f}%)")

In [68]:
count_nulls(raw_data)

pickup_datetime: 0 (0.00%)
dropoff_datetime: 0 (0.00%)
trip_distance: 0 (0.00%)
PULocationID: 0 (0.00%)
DOLocationID: 0 (0.00%)
tip_amount: 0 (0.00%)
total_amount: 0 (0.00%)
congestion_surcharge: 1,154,968 (20.50%)
cbd_congestion_fee: 5,094,435 (90.43%)


- `congestion_surcharge`: Input 2.5 if the PULocationID or DOLocationID is in CBD_CORE_ZONES, otherwise, replace null with zero.

- `cbd_congestion_fee` : Is showhing a big number of nulls. This is because this fee was introduced in `2025`, pre 2025 will be nulls. If there are nulls in 2025 observations we can replace it with zero.

`LocationID`: Decided to delete the rows where any locationID is 264 or 265 which are either "unknown" or "N/A" (Outside of NYC)

### Negative numbers

In [70]:
with_negatives = ['trip_distance', 'tip_amount', 'total_amount', 
                  'congestion_surcharge', 'cbd_congestion_fee']

def count_negatives(df:DataFrame, columns:list):
    neg_counts = {}
    for col in columns:
        if col in df.columns:
            negs = df.where(f.col(col) < 0).count()
            neg_counts[col] = negs

    for col, n in neg_counts.items():
        percent = (n / total_rows * 100) if total_rows > 0 else 0
        print(f"{col}: {n:,} ({percent:.2f}%)")

In [71]:
count_negatives(raw_data, with_negatives)

trip_distance: 822 (0.01%)
tip_amount: 778 (0.01%)
total_amount: 14,750 (0.26%)
congestion_surcharge: 195 (0.00%)
cbd_congestion_fee: 13 (0.00%)


Shouldn't be problematic if we just filter the data.

# FINAL DATA

The dataset obtained after running `src/transformations/yellowTransformations.py`

In [ ]:
final_data = spark.read.parquet(str(OUTPUT))

26/02/14 17:44:28 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/jiayi/Universidad/Tercero/2-cuatrimestre/PD2/Museekar/data/clean/clean_tlc_trip_record/green.
java.io.FileNotFoundException: File /Users/jiayi/Universidad/Tercero/2-cuatrimestre/PD2/Museekar/data/clean/clean_tlc_trip_record/green does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSo

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/jiayi/Universidad/Tercero/2-cuatrimestre/PD2/Museekar/data/clean/clean_tlc_trip_record/green. SQLSTATE: 42K03

In [ ]:
final_data.printSchema()

In [ ]:
final_data_rows = final_data.count()
total_rows - final_data_rows 

In [ ]:
print(f'Erased PCT:{(final_data_rows/total_rows*100):.2f}%')

In [ ]:
count_nulls(final_data)

In [ ]:
count_negatives(final_data, with_negatives)

In [ ]:
num_cols = ['trip_distance', 'tip_amount', 'total_amount', 
                  'congestion_surcharge', 'airport_fee', 'trip_duration']

final_summary = final_data.select(num_cols).summary().toPandas()

In [ ]:
final_summary

In [ ]:
stop(spark)